# Multi-Stage Builds

Multi-stage builds solve a fundamental tension in Dockerfiles: the tools you need to *build* an application are not the tools you need to *run* it. Compilers, build systems, test runners, and dev dependencies are needed during the build but add size and attack surface to the final image.

Before multi-stage builds, teams used separate "builder" and "runtime" Dockerfiles with shell scripts to extract artifacts. Multi-stage builds collapse this into a single Dockerfile.

Topics:
- The problem: build tools in production images
- How multi-stage builds work
- Named stages and selective targeting
- `COPY --from` — copying between stages
- Worked examples: Python, Go, Node.js
- Parallel stages and build caching
- Development vs production stages in one file

## The Problem: Build Tools in Production Images

Consider building a Go binary:

```dockerfile
# Naive approach — all build tools end up in the final image
FROM golang:1.22
WORKDIR /app
COPY . .
RUN go build -o server ./cmd/server
CMD ["/app/server"]
```

The `golang:1.22` image is ~800 MB. The compiled binary might be 15 MB. You're shipping 785 MB of build tooling that serves no purpose at runtime — and every tool in the image is a potential vulnerability.

The ideal final image contains only:
- The compiled binary (or interpreted runtime)
- The runtime dependencies it actually needs
- Nothing else

## How Multi-Stage Builds Work

A multi-stage Dockerfile has multiple `FROM` instructions. Each `FROM` starts a new stage with its own filesystem. Stages are numbered (0, 1, 2…) and can be named with `AS`.

```dockerfile
# Stage 0 — builder
FROM golang:1.22 AS builder
WORKDIR /app
COPY . .
RUN go build -o server ./cmd/server

# Stage 1 — final runtime image
FROM debian:bookworm-slim
COPY --from=builder /app/server /usr/local/bin/server
CMD ["/usr/local/bin/server"]
```

**What happens:**
1. Docker builds stage 0 (`builder`) — this has all the Go toolchain
2. Docker builds stage 1 — starts fresh from `debian:bookworm-slim`
3. `COPY --from=builder` copies *only the binary* from the builder stage
4. The final image is based on `debian:bookworm-slim` + the binary — no Go toolchain

The intermediate builder stage is discarded. It is not part of the final image.

`COPY --from` can reference:
- A stage name: `--from=builder`
- A stage index: `--from=0`
- A completely separate image: `--from=nginx:alpine` (copy a file from any image)

## Worked Example 1: Go Binary → Scratch

Go produces statically linked binaries that need no operating system — you can run them from `scratch`.

In [ ]:
import os, textwrap

os.makedirs("/tmp/goapp/cmd/server", exist_ok=True)

# Minimal Go HTTP server
with open("/tmp/goapp/cmd/server/main.go", "w") as f:
    f.write(textwrap.dedent("""\
        package main

        import (
            "fmt"
            "net/http"
        )

        func main() {
            http.HandleFunc("/", func(w http.ResponseWriter, r *http.Request) {
                fmt.Fprintln(w, "Hello from a tiny Go container!")
            })
            http.ListenAndServe(":8080", nil)
        }
    """))

with open("/tmp/goapp/go.mod", "w") as f:
    f.write("module example.com/goapp\n\ngo 1.22\n")

# Multi-stage Dockerfile: build with Go toolchain, run from scratch
with open("/tmp/goapp/Dockerfile", "w") as f:
    f.write(textwrap.dedent("""\
        # ── Stage 1: Build ───────────────────────────────────────────────────────────
        FROM golang:1.22-bookworm AS builder
        WORKDIR /app
        COPY go.mod .
        # go mod download cached separately from source
        RUN go mod download
        COPY . .
        # CGO_ENABLED=0: static binary, no libc dependency
        # -ldflags "-s -w": strip debug symbols — smaller binary
        RUN CGO_ENABLED=0 GOOS=linux go build -ldflags "-s -w" -o server ./cmd/server

        # ── Stage 2: Runtime — only the binary ───────────────────────────────────────
        FROM scratch
        COPY --from=builder /app/server /server
        EXPOSE 8080
        CMD ["/server"]
    """))

print("Go app files written")

In [ ]:
# Build and compare sizes
!docker build -t goapp:multistage /tmp/goapp/
!docker pull --quiet golang:1.22-bookworm

print("\nSize comparison:")
!docker images --format 'table {{.Repository}}:{{.Tag}}\t{{.Size}}' | grep -E 'golang:1.22|goapp'

In [ ]:
# Run it
!docker run -d --name goserver -p 8080:8080 goapp:multistage

import time; time.sleep(1)
!curl -s http://localhost:8080/

In [ ]:
# The scratch image has no shell — exec will fail
# This is intentional: no shell = no easy foothold for an attacker
!docker exec goserver sh 2>&1 || echo "No shell in scratch image — as expected"

!docker rm -f goserver
!docker rmi goapp:multistage

## Worked Example 2: Python — Separate Build and Runtime

Python isn't compiled to a native binary, but you still benefit from multi-stage builds: install dependencies (which may require build tools like `gcc` for C extensions) in a build stage, then copy only the installed packages into a clean runtime image.

In [ ]:
os.makedirs("/tmp/pyapp", exist_ok=True)

with open("/tmp/pyapp/requirements.txt", "w") as f:
    f.write("flask==3.0.3\ngunicorn==22.0.0\n")

with open("/tmp/pyapp/app.py", "w") as f:
    f.write(textwrap.dedent("""\
        from flask import Flask
        app = Flask(__name__)

        @app.route("/")
        def index():
            return "Hello from multi-stage Python!\\n"
    """))

with open("/tmp/pyapp/Dockerfile", "w") as f:
    f.write(textwrap.dedent("""\
        # ── Stage 1: Install dependencies ────────────────────────────────────────────
        # Use the full image here if any package needs compilation (gcc, make, etc.)
        FROM python:3.12-slim AS builder
        WORKDIR /install
        COPY requirements.txt .
        # Install into /install/packages — isolated from the system Python
        RUN pip install --no-cache-dir --prefix=/install/packages -r requirements.txt

        # ── Stage 2: Runtime image ────────────────────────────────────────────────────
        FROM python:3.12-slim
        ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1
        # Copy only the installed packages from the builder
        COPY --from=builder /install/packages /usr/local
        WORKDIR /app
        COPY app.py .
        EXPOSE 8000
        CMD ["gunicorn", "--bind", "0.0.0.0:8000", "app:app"]
    """))

!docker build -t pyapp:multistage /tmp/pyapp/
!docker images pyapp:multistage

## Worked Example 3: Node.js — Build Frontend Assets

A common pattern for frontend apps: use Node to compile/bundle assets, then serve them with Nginx — no Node runtime in the final image.

In [ ]:
node_dockerfile = '''\
# ── Stage 1: Build ───────────────────────────────────────────────────────────
FROM node:20-slim AS build
WORKDIR /app
COPY package*.json ./
RUN npm ci --only=production=false   # install dev deps for the build step
COPY . .
RUN npm run build                    # outputs to /app/dist

# ── Stage 2: Serve with Nginx ─────────────────────────────────────────────────
FROM nginx:alpine
# Copy built static assets from the Node build stage
COPY --from=build /app/dist /usr/share/nginx/html
# Optional: custom nginx config
# COPY nginx.conf /etc/nginx/conf.d/default.conf
EXPOSE 80
# nginx:alpine already has a CMD — no need to set one
'''
print(node_dockerfile)

The final image is `nginx:alpine` (~10 MB) + the compiled static files. Node.js, npm, and all build dependencies are gone.

## Targeting a Specific Stage

You can stop the build at a specific stage with `--target`. This is how you use one Dockerfile for both development and production.

```bash
# Build only the builder stage (useful for running tests in CI)
docker build --target builder -t myapp:test .

# Build the full production image
docker build -t myapp:prod .
```

### Dev/prod in one Dockerfile

In [ ]:
devprod_dockerfile = '''\
# syntax=docker/dockerfile:1

# ── Base: shared setup ────────────────────────────────────────────────────────
FROM python:3.12-slim AS base
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# ── Dev: adds dev tools on top of base ────────────────────────────────────────
FROM base AS dev
COPY requirements-dev.txt .
RUN pip install --no-cache-dir -r requirements-dev.txt
COPY . .
CMD ["flask", "--debug", "run", "--host", "0.0.0.0"]

# ── Test: run the test suite ──────────────────────────────────────────────────
FROM dev AS test
RUN pytest tests/ --tb=short

# ── Production: clean runtime image ──────────────────────────────────────────
FROM base AS production
RUN groupadd --gid 1001 appgroup \\
 && useradd --uid 1001 --gid appgroup --shell /bin/sh appuser
COPY --chown=appuser:appgroup . .
USER appuser
EXPOSE 8000
CMD ["gunicorn", "--bind", "0.0.0.0:8000", "app:create_app()"]
'''
print(devprod_dockerfile)

In [ ]:
# Usage:
usage = '''\
# Local development
docker build --target dev -t myapp:dev .
docker run -v $(pwd):/app -p 5000:5000 myapp:dev

# CI — run tests (stops at the test stage)
docker build --target test -t myapp:test .

# Production build — only the production stage ends up in the image
docker build --target production -t myapp:prod .
docker push myapp:prod
'''
print(usage)

## Parallel Stages with BuildKit

When you have multiple independent builder stages, BuildKit runs them in parallel. This can significantly speed up complex builds.

```dockerfile
# syntax=docker/dockerfile:1

# These two stages are independent — BuildKit runs them in parallel
FROM node:20-slim AS js-builder
WORKDIR /app
COPY package*.json .
RUN npm ci
COPY frontend/ .
RUN npm run build

FROM golang:1.22 AS go-builder
WORKDIR /app
COPY go.mod go.sum .
RUN go mod download
COPY backend/ .
RUN CGO_ENABLED=0 go build -o api ./cmd/api

# Final image assembles artifacts from both builders
FROM debian:bookworm-slim
COPY --from=go-builder  /app/api           /usr/local/bin/api
COPY --from=js-builder  /app/dist          /var/www/html
CMD ["/usr/local/bin/api"]
```

Enable BuildKit (it's the default in Docker Desktop and Docker Engine 23+):
```bash
DOCKER_BUILDKIT=1 docker build -t myapp .
# or via the first line: # syntax=docker/dockerfile:1
```

## `COPY --from` an External Image

`--from` doesn't have to reference a stage in the current Dockerfile — you can copy from any image:

```dockerfile
# Copy a specific binary from an official image without adding that image as a base
COPY --from=golang:1.22 /usr/local/go/bin/go /usr/local/bin/go

# Pull in a CA certificate bundle from a known-good image
COPY --from=alpine:3.19 /etc/ssl/certs/ca-certificates.crt /etc/ssl/certs/
```

This is useful for:
- Pulling in a single tool without inheriting its full image
- Getting SSL certificates into a `scratch`-based image
- Injecting a binary from a versioned official image

In [ ]:
# Practical scratch + CA certs example — Go binary that makes HTTPS requests
scratch_tls_dockerfile = '''\
FROM golang:1.22-bookworm AS builder
WORKDIR /app
COPY . .
RUN CGO_ENABLED=0 go build -ldflags "-s -w" -o myapp .

FROM scratch
# Without these, HTTPS requests fail: no CA bundle in scratch
COPY --from=alpine:3.19 /etc/ssl/certs/ca-certificates.crt /etc/ssl/certs/
COPY --from=builder /app/myapp /myapp
CMD ["/myapp"]
'''
print(scratch_tls_dockerfile)

In [ ]:
# Clean up images from earlier examples
!docker rmi pyapp:multistage 2>/dev/null || true
!echo "Cleanup done"

## Summary

- **Multi-stage builds** use multiple `FROM` instructions in one Dockerfile. Each stage has its own filesystem. Only the final stage becomes the image — intermediate stages are discarded.
- **`COPY --from=stage`** copies files from an earlier stage (or any image) into the current stage. This is how you extract a compiled binary from a heavy build environment into a lean runtime image.
- **Name stages** with `AS name` — more readable and stable than numeric indexes.
- **`--target stage`** stops the build at a named stage. Use this to share one Dockerfile between dev, test, and production targets.
- **Parallel stages** — BuildKit runs independent stages concurrently. Add `# syntax=docker/dockerfile:1` to enable it.
- **Go → scratch** is the gold standard for minimal images. A statically compiled binary with no OS overhead.
- **Python/Node** still benefit from multi-stage builds: compile C extensions or bundle assets in a builder, copy the results to a clean runtime image.
- **`COPY --from=image`** can pull files from any Docker image, not just stages in the current Dockerfile — useful for CA certificates in `scratch` images or injecting a single binary.